In [6]:
import pandas as pd
df_movie = pd.read_excel('movies.xlsx')
df_ott = pd.read_csv('ott.csv')
print(df_movie.head())
print(df_movie.shape)
print(df_ott.head())
print(df_ott.shape)

                            Title  Year  Age  IMDb  Rotten Tomatoes  \
0                       Inception  2010  13+   8.8             0.87   
1                      The Matrix  1999  18+   8.7             0.87   
2          Avengers: Infinity War  2018  13+   8.5             0.84   
3              Back to the Future  1985   7+   8.5             0.96   
4  The Good, the Bad and the Ugly  1966  18+   8.8             0.97   

                        Directors                            Genres  \
0               Christopher Nolan  Action,Adventure,Sci-Fi,Thriller   
1  Lana Wachowski,Lilly Wachowski                     Action,Sci-Fi   
2         Anthony Russo,Joe Russo           Action,Adventure,Sci-Fi   
3                 Robert Zemeckis           Adventure,Comedy,Sci-Fi   
4                    Sergio Leone                           Western   

                        Country                 Language  Runtime  
0  United States,United Kingdom  English,Japanese,French    148.0  
1         

In [9]:
print(df_movie.dtypes)
print(df_ott.dtypes)

Title               object
Year                 int64
Age                    str
IMDb               float64
Rotten Tomatoes    float64
Directors              str
Genres                 str
Country                str
Language               str
Runtime            float64
dtype: object
ID             int64
Netflix        int64
Stan           int64
Prime Video    int64
Disney+        int64
dtype: object


In [11]:
print(df_ott['ID'].is_unique)                                   # should be True
print(df_ott['ID'].min(), df_ott['ID'].max())                   # should be 1 16744
print((df_ott['ID'] == range(1, len(df_ott) + 1)).all())        # True = in order, no gaps

True
1 16744
True


In [12]:
df_movie['ID'] = range(1, len(df_movie) + 1)

df_merged = pd.merge(df_movie, df_ott, on='ID', how='left', indicator=True)

print(df_merged['_merge'].value_counts())   # all 16744 should be 'both'
print(df_merged.shape)                      # expect (16744, 16): 10 + ID + 4 platforms + _merge
print(df_merged.head())

df_merged = df_merged.drop(columns='_merge')

_merge
both          16744
left_only         0
right_only        0
Name: count, dtype: int64
(16744, 16)
                            Title  Year  Age  IMDb  Rotten Tomatoes  \
0                       Inception  2010  13+   8.8             0.87   
1                      The Matrix  1999  18+   8.7             0.87   
2          Avengers: Infinity War  2018  13+   8.5             0.84   
3              Back to the Future  1985   7+   8.5             0.96   
4  The Good, the Bad and the Ugly  1966  18+   8.8             0.97   

                        Directors                            Genres  \
0               Christopher Nolan  Action,Adventure,Sci-Fi,Thriller   
1  Lana Wachowski,Lilly Wachowski                     Action,Sci-Fi   
2         Anthony Russo,Joe Russo           Action,Adventure,Sci-Fi   
3                 Robert Zemeckis           Adventure,Comedy,Sci-Fi   
4                    Sergio Leone                           Western   

                        Country          

In [13]:
pd.concat([df_movie, df_ott], axis=0)

,Title,Year,Age,IMDb,Rotten Tomatoes,Directors,Genres,Country,Language,Runtime,ID,Netflix,Stan,Prime Video,Disney+
0,Inception,2010.0,13+,8.8,0.87,Christopher Nolan,"Action,Adventure,Sci-Fi,Thriller","United States,United Kingdom","English,Japanese,French",148.0,1,NaN,NaN,NaN,NaN
1,The Matrix,1999.0,18+,8.7,0.87,"Lana Wachowski,Lilly Wachowski","Action,Sci-Fi",United States,English,136.0,2,NaN,NaN,NaN,NaN
2,Avengers: Infinity War,2018.0,13+,8.5,0.84,"Anthony Russo,Joe Russo","Action,Adventure,Sci-Fi",United States,English,149.0,3,NaN,NaN,NaN,NaN
3,Back to the Future,1985.0,7+,8.5,0.96,Robert Zemeckis,"Adventure,Comedy,Sci-Fi",United States,English,116.0,4,NaN,NaN,NaN,NaN
4,"The Good, the Bad and the Ugly",1966.0,18+,8.8,0.97,Sergio Leone,Western,"Italy,Spain,West Germany",Italian,161.0,5,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16739,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16740,0.0,0.0,1.0,0.0
16740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16741,0.0,0.0,1.0,0.0
16741,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16742,0.0,0.0,1.0,0.0
16742,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16743,0.0,0.0,1.0,0.0


In [27]:
# movies per year available on netflix
mo_gpby = (
    df_merged.groupby(['Netflix', 'Year'])
        .agg({'Title':'count'})
    .reset_index()
    .rename(columns={'Title': 'Films'})
)

netflix_2012 = mo_gpby[(mo_gpby['Year'] >= 2012) & (mo_gpby['Netflix'] == 1)]

netflix_2012

,Netflix,Year,Films
199,1,2012,195
200,1,2013,200
201,1,2014,202
202,1,2015,252
203,1,2016,236
204,1,2017,294
205,1,2018,307
206,1,2019,138
207,1,2020,31


"New movies in Netflix has peaked 2017, appears to be tapering off"

In [16]:
mo_runtime = (
    df_merged
    .groupby('Year')
    .agg({'Runtime':'mean'})
    .reset_index()
)
mo_runtime = mo_runtime[mo_runtime['Year'] >=2012]
mo_runtime

,Year,Runtime
100,2012,90.511714
101,2013,91.545940
102,2014,92.995763
103,2015,92.914646
104,2016,93.992140
105,2017,94.460961
106,2018,94.635678
107,2019,93.410413
108,2020,93.976562


"industry seems pretty static with run time, very slight increases may be a result of invcreased advertising/marketing but only an assumption"

In [22]:
mo_reviews = (
    df_merged
    .groupby('Year')
    .agg({'Rotten Tomatoes': ['min', 'max']})
    .reset_index()
)

mo_reviews = mo_reviews[mo_reviews['Year'] >= 2012]

mo_reviews

Year Rotten Tomatoes     
                      min  max
100  2012            0.04  1.0
101  2013            0.02  1.0
102  2014            0.05  1.0
103  2015            0.05  1.0
104  2016            0.02  1.0
105  2017            0.04  1.0
106  2018            0.06  1.0
107  2019            0.05  1.0
108  2020            0.06  1.0

"Films appear to be establishing a better base line in recent years, its a small sample but this may indicate a drop in quality in the coming year"

Scenario
Continuing Mandisa Nkosi's analysis: The political party is considering advertisements with provocative messages and needs to ensure content suits the audiences watching. The data includes content ratings (the Age column) that Mandisa can use to organise the DataFrame.

In [28]:
rating_by_year = pd.pivot_table(
    df_merged,
    index='Year',
    columns='Age',
    values='ID',
    aggfunc='count',
    fill_value=0
)

rating_by_year

Age,13+,16+,18+,7+,all
Year,,,,,
1902,0,0,0,0,1
1912,0,0,0,1,0
1915,0,0,0,1,0
1919,1,0,0,0,0
1920,0,0,0,1,0
...,...,...,...,...,...
2016,92,32,184,95,37
2017,91,63,270,98,40
2018,92,42,255,87,23


In [29]:
# average runtime by rating and year: index Year; columns Age; values Runtime;
# many films share each Year–Age combination, so their runtimes have to be combined into one number.
# This time I took the average instead of counting.

runtime_by_rating = pd.pivot_table(
    df_merged,
    index='Year',
    columns='Age',
    values='Runtime',
    aggfunc='mean'
).round(1)

runtime_by_rating.loc[2012:]

Age,13+,16+,18+,7+,all
Year,,,,,
2012,103.2,84.2,90.4,97.9,78.6
2013,99.4,86.2,92.9,97.2,94.9
2014,105.0,90.4,97.2,95.3,94.6
2015,101.2,93.3,94.9,95.5,90.8
2016,102.0,97.9,95.2,92.6,84.3
2017,101.8,91.6,97.6,98.2,79.2
2018,101.4,97.9,98.4,96.9,91.9
2019,103.9,98.9,99.3,93.7,93.0
2020,93.9,95.7,101.5,91.1,42.0


In [30]:
# fix the columns first to account for the multiple languages listed for a single movie to avoid exces rows being created

df_merged['Main_Language'] = df_merged['Language'].str.split(',').str[0]

In [31]:
rt_by_language = pd.pivot_table(
    df_merged,
    index='Main_Language',
    columns='Age',
    values='Rotten Tomatoes',
    aggfunc='mean'
)

rt_by_language = (rt_by_language * 100).round(0)   # 0.87 -> 87
rt_by_language

Age,13+,16+,18+,7+,all
Main_Language,,,,,
Afrikaans,75.0,NaN,NaN,NaN,NaN
American Sign Language,46.0,NaN,NaN,NaN,NaN
Arabic,76.0,90.0,46.0,95.0,NaN
Aramaic,NaN,NaN,34.0,NaN,NaN
Basque,NaN,NaN,NaN,NaN,20.0
Cantonese,87.0,NaN,85.0,33.0,NaN
Catalan,NaN,NaN,11.0,NaN,NaN
Chinese,83.0,NaN,NaN,55.0,40.0
Danish,NaN,86.0,84.0,43.0,44.0


In [32]:
# True false is on netflex?

netflix = df_merged[df_merged['Netflix'] == 1]

print(f"Netflix films: {len(netflix):,} of {len(df_merged):,}")

Netflix films: 3,560 of 16,744


pivot() doesn't aggregate. It will only move each value into its place in the grid, so every index and column pair must appear exactly once. The raw data doesn't really work for this, because a single year has many films rated 13+. A summary table should:  groupby Year and Age pair to result in one row (fingers crossed)

In [33]:
# Build a subset where each Year–Age pair appears once
runtime_long = (
    df_merged
    .groupby(['Year', 'Age'])
    .agg({'Runtime': 'mean'})
    .reset_index()
)

runtime_long = runtime_long[runtime_long['Year'] >= 2012]
runtime_long.head()

,Year,Age,Runtime
280,2012,13+,103.213333
281,2012,16+,84.190476
282,2012,18+,90.376712
283,2012,7+,97.915254
284,2012,all,78.558824


In [34]:
print("Duplicate Year–Age pairs:", runtime_long.duplicated(subset=['Year', 'Age']).sum())

Duplicate Year–Age pairs: 0


In [35]:
runtime_wide = runtime_long.pivot(index='Year', columns='Age', values='Runtime').round(1)
runtime_wide

Age,13+,16+,18+,7+,all
Year,,,,,
2012,103.2,84.2,90.4,97.9,78.6
2013,99.4,86.2,92.9,97.2,94.9
2014,105.0,90.4,97.2,95.3,94.6
2015,101.2,93.3,94.9,95.5,90.8
2016,102.0,97.9,95.2,92.6,84.3
2017,101.8,91.6,97.6,98.2,79.2
2018,101.4,97.9,98.4,96.9,91.9
2019,103.9,98.9,99.3,93.7,93.0
2020,93.9,95.7,101.5,91.1,42.0


In [36]:
print("Matches pivot_table version:", runtime_wide.equals(runtime_by_rating.loc[2012:]))
print("Cells filled:", runtime_wide.notna().sum().sum(), "| rows in long table:", len(runtime_long))

Matches pivot_table version: True
Cells filled: 45 | rows in long table: 45


The first pivot, film counts by rating and year, is the most useful of the five. Even so, it only gives the party background, not an answer.

The party's question is whether provocative messages suit the audience. Age rating is the only column in this dataset related to that. The first pivot shows the rating mix: how much of the content on these platforms is rated 18+ compared with 7+ or 13+, and whether that mix is changing. If the recent catalogue leans heavily towards 18+, the platforms are clearly aimed at adult viewers, which supports running bolder messages there. If it leans towards family ratings, a provocative ad is more likely to reach children and to cause offence.